In [0]:
from pyspark.sql import functions as F

caminho_volume = "/Volumes/prf_acidentes/bronze/raw_files/"

arquivos = {
    2023: "datatran2023.csv",
    2024: "datatran2024.csv",
    2025: "datatran2025.csv"
}

dataframes = []

for ano, nome_arquivo in arquivos.items():
    df = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .option("inferSchema", "false")  # <-- mudou: tudo como string
        .csv(caminho_volume + nome_arquivo)
        .withColumn("_ano_referencia", F.lit(ano))
        .withColumn("_arquivo_origem", F.lit(nome_arquivo))
        .withColumn("_data_ingestao", F.current_timestamp())
    )
    dataframes.append(df)

df_bronze = dataframes[0]
for df in dataframes[1:]:
    df_bronze = df_bronze.unionByName(df, allowMissingColumns=True)

print("Total de linhas:", df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("prf_acidentes.bronze.acidentes_raw")

print("Tabela Bronze criada com sucesso!")

In [0]:
display(spark.sql("SELECT * FROM prf_acidentes.bronze.acidentes_raw LIMIT 10"))

In [0]:
display(spark.sql("SHOW TABLES IN prf_acidentes.bronze"))

In [0]:
display(spark.sql("DESCRIBE TABLE prf_acidentes.bronze.acidentes_raw"))